# Woche 6 – Mittwoch: Lp‑Räume (Elstrodt Kap. 4.4) & VaR‑Vertiefung

## Block 1 (06:00 – 08:45): Lp‑Räume

**Lernziel:** Sie definieren die Räume $L^p$ als Vervollständigung der integrierbaren Funktionen.

Für $1 \le p < \infty$ ist $L^p(\mathbb{R}^n) = \{ f \text{ messbar} : \int |f|^p \, d\lambda < \infty \}$ mit Norm $\|f\|_p = (\int |f|^p)^{1/p}$.

$L^\infty$ besteht aus den wesentlich beschränkten Funktionen mit $\|f\|_\infty = \inf\{ C : |f(x)| \le C \text{ f.ü.} \}$.

**Aufgabe:** Zeigen Sie, dass $L^p \subset L^q$ für $1 \le p \le q \le \infty$ auf endlichen Maßräumen gilt (z. B. auf $[0,1]$).

> **Verbesserungshinweis:** Die Lp‑Räume sind Banachräume. Für $p=2$ erhält man einen Hilbertraum – das ist wichtig für PCA und Fourier‑Analyse.

---

## Block 2 (09:30 – 11:40): Anomalieerkennung mit Autoencoder – Rekonstruktionsfehler

**Lernziel:** Sie berechnen den Rekonstruktionsfehler und identifizieren Ausreißer.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Daten laden und vorbereiten
df = pd.read_json('cleaned_aml_batch.json')
features = df[['amount']].copy()
features['amount_log'] = np.log1p(features['amount'])
scaler = StandardScaler()
X = torch.tensor(scaler.fit_transform(features), dtype=torch.float32)

# Autoencoder (wie Montag)
class AE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(2, 1)
        self.decoder = nn.Linear(1, 2)
    def forward(self, x):
        return self.decoder(self.encoder(x))

model = AE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# Training
for epoch in range(200):
    out = model(X)
    loss = criterion(out, X)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Rekonstruktionsfehler berechnen
model.eval()
with torch.no_grad():
    reconstructions = model(X)
    errors = torch.mean((X - reconstructions)**2, dim=1).numpy()
df['ae_error'] = errors
top_anomalies = df.sort_values('ae_error', ascending=False).head(5)
print(top_anomalies[['amount', 'ae_error']])


> **Verbesserungshinweis:** Hoher Fehler deutet auf eine ungewöhnliche Transaktion hin – in der Praxis würden diese manuell geprüft.

---

## Block 3 (12:40 – 14:10): GOV – Wiederholung Art. 16‑21

Erstellen Sie in GOV eine **Checkliste für den Anbieter** und eine separate für den **Betreiber** basierend auf Art. 16‑21. Welche Punkte sind für Ihr AML‑System besonders relevant?

> **Verbesserungshinweis:** Die Unterscheidung Anbieter (entwickelt das System) vs. Betreiber (nutzt es) ist entscheidend für die Zuordnung der Pflichten.

---

## Block 4 (14:40 – 16:50): Passiv mit Modalverben (Buscha Kap. 1.5.1 Vertiefung)

**Übung:** Wandeln Sie folgende Sätze ins Passiv mit Modalverb um („können“, „müssen“, „sollen“):
1. Der Anbieter muss die Konformitätsbewertung durchführen.
2. Die Bank kann das System täglich überwachen.
3. Die Behörde soll die Dokumentation prüfen.

**Schreibübung:** Verfassen Sie eine kurze Anleitung (5‑10 Sätze) für den Betrieb des Autoencoders, ausschließlich im Passiv mit Modalverben. Speichern Sie als `25_betriebsanleitung_passiv.md` in Track_C.

> **Verbesserungshinweis:** Passiv mit Modalverb: „Die Konformitätsbewertung muss vom Anbieter durchgeführt werden.“